# med-data-EDA — PMC 数据探索与分析

阶段 02：数据加载、字段统计、长度分布、分割策略（见 `schedule.md`）。

**运行环境**：VS Code 用 **Open Folder → `02 数据处理/`**，内核 **`med-rag-verify`**。

---

## ⚠️ 每次重新打开 notebook 请先运行

| 顺序 | Cell 标题 | 作用 |
|---|---|---|
| **1** | 【前置 1/2】 | 路径、`importlib.reload`、**全部 import** |
| **2** | 【前置 2/2】 | 加载 `ds` / `df` / `df_clean` |

| 章节 | 状态 |
|---|---|
| §3 | ✅ 已完成 |
| §4 | ✅ 领域理解（分层 + IMRaD/缩写） |
| §5 | ✅ token 分位数 + ECDF |
| §6 | ✅ 分割策略定稿 + demo |
| **文末** | ✅ [验证期最终策略摘要](#小规模数据集验证阶段最终策略)（§4～§6 结论浓缩） |

跑 §4~§6 前必先跑前置；其它章节**不要**再写 `from load_pipeline import ...`。

可选：菜单 **Run All** 会按顺序跑完全部 cell。

## 【前置 1/2】环境与路径（每次打开必跑）

`importlib.reload` 确保 `src/load_pipeline.py` 的修改生效（避免 `TITLE_SUSPICIOUS_LEN` 等 ImportError）。

In [4]:
import importlib
import os
import sys


def _bootstrap_src() -> str:
    """从 CWD 向上找含 任务.txt 的工程根，并把 src/ 加入 sys.path。"""
    cur = os.path.abspath(os.getcwd())
    while True:
        marker = os.path.join(cur, "任务.txt")
        if os.path.isfile(marker):
            src = os.path.join(cur, "src")
            if src not in sys.path:
                sys.path.insert(0, src)
            return cur
        parent = os.path.dirname(cur)
        if parent == cur:
            break
        cur = parent
    raise RuntimeError(
        "未能定位工程根目录。请在 VS Code 用 File → Open Folder 打开「02 数据处理」。"
    )


PROJECT_DIR = _bootstrap_src()
os.environ.setdefault("PROJECT_DIR", PROJECT_DIR)

# 重载模块（内核会缓存旧版 load_pipeline，导致新增常量 import 失败）
import load_pipeline as _load_pipeline

importlib.reload(_load_pipeline)

from load_pipeline import (  # noqa: E402
    ABSTRACT_MISSING_RATE_ALERT,
    BODY_MIN_CHAR_THRESHOLD,
    EXPECTED_COLUMNS,
    FIELD_MAP,
    RETRIEVAL_FIELDS,
    SHORT_ABSTRACT_CHAR_THRESHOLD,
    TITLE_SUSPICIOUS_LEN,
    TOKENIZER_MAX_LENGTH,
    TOKENIZER_MODEL_ID,
    describe_dataset,
    drop_missing_abstract,
    field_completeness_table,
    load_pmc_jsonl,
    metadata_summary,
    print_config_summary,
    quality_flags_table,
    resolve_project_dir,
    setup_paths,
    validate_schema,
)

PROJECT_DIR, PROJECT_DIR_SRC = resolve_project_dir()
PATHS = setup_paths(PROJECT_DIR)
SAMPLE_JSONL = PATHS["sample_jsonl"]
TABLES_DIR = PATHS["tables_dir"]
CLEAN_JSONL = os.path.join(PATHS["data_processed"], "sample_clean.jsonl")

print(f"PROJECT_DIR 来源: {PROJECT_DIR_SRC}")
print_config_summary(PROJECT_DIR, PATHS)
print("\n预期 jsonl 列:", EXPECTED_COLUMNS)
print(f"\n[OK] 前置 1/2 — load_pipeline 已 reload，TITLE_SUSPICIOUS_LEN={TITLE_SUSPICIOUS_LEN}")

PROJECT_DIR 来源: 环境变量 PROJECT_DIR
=== 阶段 0：概念与指标约定 ===
PROJECT_DIR     : /Users/sanxue/Desktop/work/实习/谷歌/02 数据处理
DATA_ROOT       : /Users/sanxue/Desktop/work/实习/谷歌/02 数据处理/data
SAMPLE_JSONL    : /Users/sanxue/Desktop/work/实习/谷歌/02 数据处理/data/processed/sample.jsonl
Tokenizer（统计）: sentence-transformers/all-MiniLM-L6-v2 (max 512 tokens)
检索单元字段    : title + abstract → retrieval_text
任务书 pub_date  : 映射为 jsonl 列「pub_date」
任务书 pmid      : jsonl 列「pmid」（由 02 parse_pmc 抽取）
清洗阈值（初稿）: abstract 缺失率 > 1% 需决策；abstract < 50 字符标记 is_short_abstract

预期 jsonl 列: ['pmcid', 'pmid', 'title', 'abstract', 'body', 'journal', 'pub_year', 'pub_date', 'n_chars_abstract', 'n_chars_body']

[OK] 前置 1/2 — load_pipeline 已 reload，TITLE_SUSPICIOUS_LEN=500


## 【前置 2/2】加载数据集（每次打开必跑）

生成全局变量：`ds`（datasets）、`df`（pandas）、`df_clean`（丢弃无 abstract 后 97 篇）。

In [5]:
import os

import pandas as pd
from datasets import Dataset
from IPython.display import display

assert os.path.isfile(SAMPLE_JSONL), f"缺少: {SAMPLE_JSONL}"

ds: Dataset = load_pmc_jsonl(SAMPLE_JSONL, backend="datasets", add_derived=True)
df: pd.DataFrame = ds.to_pandas()
df_clean: pd.DataFrame = drop_missing_abstract(df)

print(f"[OK] 前置 2/2 — ds/df: {len(df)} 篇 | df_clean: {len(df_clean)} 篇")
print(f"     SAMPLE_JSONL = {SAMPLE_JSONL}")

[OK] 前置 2/2 — ds/df: 100 篇 | df_clean: 97 篇
     SAMPLE_JSONL = /Users/sanxue/Desktop/work/实习/谷歌/02 数据处理/data/processed/sample.jsonl


## §0 / §2 参考输出（可选，前置已加载时可跳过）

### §1.5 从 XML 构建 JSONL（02 标准，不依赖 01）

验证期 XML 默认自动使用 `01 验证模型/data/raw/extracted/`（也可设 `PMC_XML_ROOT`）。全量期指向外接盘 `extracted/` 后**同一脚本**即可。

```bash
# 在 02 数据处理/ 下
./scripts/build_jsonl.sh --pmcids-from data/processed/sample.jsonl.bak01   # 对齐 100 pmcid
# ./scripts/build_jsonl.sh --limit 100
# PMC_XML_ROOT=/Volumes/你的盘/med-rag-pmc/extracted ./scripts/build_jsonl.sh -o data/processed/oa_comm_full.jsonl
```

## §2 阶段 2：数据加载 pipeline

使用 `datasets.load_dataset("json", ...)` 作为主后端；并用 `pandas` 读同一文件做行数交叉验证。

In [15]:
# 可选：在 notebook 内触发重构建（通常命令行跑即可）
import os
import subprocess

_rebuild = False  # 改为 True 可重新从 XML 生成 sample.jsonl
if _rebuild:
    script = os.path.join(PROJECT_DIR, "scripts", "build_jsonl.sh")
    subprocess.run(
        [script, "--pmcids-from", "data/processed/sample.jsonl.bak01"],
        check=True,
        cwd=PROJECT_DIR,
    )
    print("已重生成 sample.jsonl")
else:
    print("跳过重构建（_rebuild=False）。命令行: ./scripts/build_jsonl.sh")

跳过重构建（_rebuild=False）。命令行: ./scripts/build_jsonl.sh


In [16]:
# 依赖【前置 2/2】已加载的 ds
schema = validate_schema(ds)
print("=== Schema 检查 ===")
print(f"行数: {schema['n_rows']}")
print(f"列: {schema['columns']}")
print(f"缺少预期列: {schema['missing_expected'] or '无'}")
print(f"额外列: {schema['extra_columns'] or '无'}")
print(f"pmcid 唯一: {schema['pmcid_unique']} (重复数={schema['duplicate_pmcid']})")

=== Schema 检查 ===
行数: 100
列: ['pmcid', 'pmid', 'title', 'abstract', 'body', 'journal', 'pub_year', 'pub_date', 'n_chars_abstract', 'n_chars_body', 'has_abstract', 'has_body', 'title_char_len', 'abstract_char_len', 'body_char_len', 'retrieval_text', 'is_short_abstract']
缺少预期列: 无
额外列: 无
pmcid 唯一: True (重复数=0)


In [17]:
facts = describe_dataset(ds, preview_n=2)

print("=== 数据集事实（写入说明文档 §数据集事实）===")
print(f"样本文件: {SAMPLE_JSONL}")
print(f"文档数: {facts['n_rows']}")
print(f"abstract 缺失率: {facts['abstract_missing_rate']:.2%}")
print(f"title 缺失率: {facts['title_missing_rate']:.2%}")
print(f"含 abstract 比例: {facts['has_abstract_rate']:.2%}")

if facts["abstract_missing_rate"] > ABSTRACT_MISSING_RATE_ALERT:
    print(
        f"⚠ abstract 缺失率 > {ABSTRACT_MISSING_RATE_ALERT:.0%}，"
        "请在阶段 3 说明文档中确定丢弃或填充策略。"
    )
else:
    print(
        f"✓ abstract 缺失率未超过 {ABSTRACT_MISSING_RATE_ALERT:.0%} 告警线（最终以 §3 完整表为准）。"
    )

short_n = sum(ds["is_short_abstract"])
print(f"极短 abstract (<{SHORT_ABSTRACT_CHAR_THRESHOLD} 字符): {short_n} 篇")

print("\n=== 派生列样例（retrieval_text 前 200 字符）===")
preview = facts["preview"].to_pandas()
cols = ["pmcid", "journal", "pub_year", "has_abstract", "abstract_char_len", "body_char_len"]
display(preview[cols])

for i, row in preview.iterrows():
    text = row["retrieval_text"][:200].replace("\n", " ")
    print(f"\n[{row['pmcid']}] retrieval_text[:200]: {text}...")

=== 数据集事实（写入说明文档 §数据集事实）===
样本文件: /Users/sanxue/Desktop/work/实习/谷歌/02 数据处理/data/processed/sample.jsonl
文档数: 100
abstract 缺失率: 3.00%
title 缺失率: 0.00%
含 abstract 比例: 97.00%
⚠ abstract 缺失率 > 1%，请在阶段 3 说明文档中确定丢弃或填充策略。
极短 abstract (<50 字符): 0 篇

=== 派生列样例（retrieval_text 前 200 字符）===


,pmcid,journal,pub_year,has_abstract,abstract_char_len,body_char_len
0,PMC8774754,Diagnostics,2022,True,1243,13813
1,PMC8947183,Diagnostics,2022,True,1386,21479



[PMC8774754] retrieval_text[:200]: Axonemal Symmetry Break, a New Ultrastructural Diagnostic Tool for Primary Ciliary Dyskinesia? Diagnosis testing for primary ciliary dyskinesia (PCD) requires a combination of investigations that incl...

[PMC8947183] retrieval_text[:200]: Diffusion Weighted Imaging in the Assessment of Tumor Grade in Endometrial Cancer Based on Intravoxel Incoherent Motion MRI The aim of this study is to investigate the possibility of predicting histol...


In [18]:
# 全量字符长度概览（token 分布在 §5 再算）
summary = df[[
    "title_char_len", "abstract_char_len", "body_char_len",
    "n_chars_abstract", "n_chars_body",
]].describe()
display(summary.round(1))

print("\n下一阶段: §3 字段完整性表（见下方）")

,title_char_len,abstract_char_len,body_char_len,n_chars_abstract,n_chars_body
count,100.0,100.0,100.0,100.0,100.0
mean,112.6,1709.5,39824.2,1709.5,39824.2
std,36.3,705.2,24341.4,705.2,24341.4
min,31.0,0.0,631.0,0.0,631.0
25%,89.8,1388.0,24072.5,1388.0,24072.5
50%,104.5,1624.5,31880.0,1624.5,31880.0
75%,135.0,1930.8,50805.0,1930.8,50805.0
max,254.0,5184.0,135425.0,5184.0,135425.0



下一阶段: §3 字段完整性表（见下方）


## §3 阶段 3：数据结构分析（任务 §1）

字段完整性 → 清洗策略 → 基础质量 → 元数据（journal / 年份 / pmid）。

In [19]:
# 依赖【前置 1/2】【前置 2/2】
_missing = [k for k in ("df", "ds", "TABLES_DIR", "TITLE_SUSPICIOUS_LEN") if k not in globals()]
if _missing:
    raise RuntimeError(f"请先运行【前置 1/2】和【前置 2/2】，缺少变量: {_missing}")

completeness = field_completeness_table(df)
completeness_path = os.path.join(TABLES_DIR, "field_completeness.csv")
completeness.to_csv(completeness_path, index=False)

print("=== §3.1 字段完整性 ===")
display(completeness.style.format({"missing_rate": "{:.2%}", "fill_rate": "{:.2%}"}))
print(f"\n已保存: {completeness_path}")

abstract_miss = completeness.loc[completeness["field"] == "abstract", "missing_rate"].iloc[0]
print(f"\nabstract 缺失率: {abstract_miss:.2%}（阈值 {ABSTRACT_MISSING_RATE_ALERT:.0%}）")

=== §3.1 字段完整性 ===


,field,non_null_count,missing_count,missing_rate,fill_rate
0,pmcid,100,0,0.00%,100.00%
1,pmid,93,7,7.00%,93.00%
2,title,100,0,0.00%,100.00%
3,abstract,97,3,3.00%,97.00%
4,body,100,0,0.00%,100.00%
5,journal,100,0,0.00%,100.00%
6,pub_year,100,0,0.00%,100.00%
7,pub_date,100,0,0.00%,100.00%



已保存: /Users/sanxue/Desktop/work/实习/谷歌/02 数据处理/outputs/tables/field_completeness.csv

abstract 缺失率: 3.00%（阈值 1%）


In [20]:
# §3.1 清洗策略：abstract 缺失 > 1% → 丢弃（RAG 推荐）
DROP_NO_ABSTRACT = True  # 定稿后写入说明文档

if abstract_miss > ABSTRACT_MISSING_RATE_ALERT:
    print(f"⚠ 超过阈值，采用策略: {'丢弃无 abstract' if DROP_NO_ABSTRACT else '填充占位符'}")
else:
    print("✓ 未超阈值，可保留全部记录")

if DROP_NO_ABSTRACT:
    df_clean = drop_missing_abstract(df)
    clean_path = os.path.join(PATHS["data_processed"], "sample_clean.jsonl")
    raw_cols = [c for c in EXPECTED_COLUMNS if c in df_clean.columns]
    df_clean[raw_cols].to_json(clean_path, orient="records", lines=True, force_ascii=False)
    print(f"清洗后: {len(df_clean)} 篇（丢弃 {len(df) - len(df_clean)} 篇）→ {clean_path}")
    display(df.loc[~df["has_abstract"], ["pmcid", "journal", "title_char_len"]])
else:
    df_clean = df.copy()

⚠ 超过阈值，采用策略: 丢弃无 abstract
清洗后: 97 篇（丢弃 3 篇）→ /Users/sanxue/Desktop/work/实习/谷歌/02 数据处理/data/processed/sample_clean.jsonl


/var/folders/8p/j7n2w7cs2kq712tnw3bxmf6c0000gn/T/ipykernel_15613/999161882.py:13: Pandas4Warning: The default 'epoch' date format is deprecated and will be removed in a future version, please use 'iso' date format instead.
  df_clean[raw_cols].to_json(clean_path, orient="records", lines=True, force_ascii=False)


,pmcid,journal,title_char_len
13,PMC12110669,Current Issues in Molecular Biology,67
71,PMC12882845,Metabolic Brain Disease,199
90,PMC12882890,Nuclear Medicine and Molecular Imaging,95


In [21]:
# §3.2 基础质量
quality = quality_flags_table(df)
quality_path = os.path.join(TABLES_DIR, "quality_flags.csv")
quality.to_csv(quality_path, index=False)

print("=== §3.2 基础质量 ===")
display(quality.style.format({"rate": "{:.2%}"}))
print(f"已保存: {quality_path}")

long_titles = df[df["title_char_len"] > TITLE_SUSPICIOUS_LEN]
if len(long_titles):
    print(f"\n异常长 title (>{TITLE_SUSPICIOUS_LEN} 字符): {len(long_titles)} 篇")
    display(long_titles[["pmcid", "title_char_len", "title"]].head())
else:
    print(f"\n✓ 无异常长 title (>{TITLE_SUSPICIOUS_LEN} 字符)，02 parser 正常")

dup = df["pmcid"].duplicated().sum()
print(f"重复 pmcid: {dup}")

=== §3.2 基础质量 ===


,flag,count,rate
0,missing_abstract,3,3.00%
1,short_abstract,0,0.00%
2,suspicious_long_title,0,0.00%
3,short_body,0,0.00%
4,xml_residue_abstract,0,0.00%
5,xml_residue_title,0,0.00%


已保存: /Users/sanxue/Desktop/work/实习/谷歌/02 数据处理/outputs/tables/quality_flags.csv

✓ 无异常长 title (>500 字符)，02 parser 正常
重复 pmcid: 0


In [34]:
# §3.3 元数据价值：journal / pub_year / pmid
meta = metadata_summary(df, top_journals=10)
journal_path = os.path.join(TABLES_DIR, "journal_top10.csv")
meta["top_journals"].to_csv(journal_path, index=False)

print("=== §3.3 元数据 ===")
print(f"期刊种类数: {meta['journal_nunique']}")
print(f"pmid 覆盖率: {meta['pmid_fill_rate']:.2%} ({meta['pmid_fill_count']}/{meta['n']})")
print(f"pub_year 范围: {meta['pub_year_min']} — {meta['pub_year_max']}")
print(f"近 5 年（{pd.Timestamp.now().year - 5}+）: {meta['pub_year_recent_5y_count']} 篇 ({meta['pub_year_recent_5y_rate']:.2%})")

display(meta["top_journals"])

# 年份分布
year_vc = (
    pd.to_numeric(df["pub_year"], errors="coerce")
    .value_counts()
    .sort_index()
    .reset_index()
)
year_vc.columns = ["pub_year", "count"]
year_vc.to_csv(os.path.join(TABLES_DIR, "pub_year_distribution.csv"), index=False)
display(year_vc)

print("\n--- 元数据过滤可行性（写入说明文档 §7）---")
print("· 按 journal 过滤：字段完整，但刊名未标准化（如缩写/全称），「Nature」需额外映射表。")
print("· 按 pub_year 近 5 年：pub_year 已可用；精确到月日时用 pub_date。")
print("· pmid 追溯：约", f"{meta['pmid_fill_rate']:.0%}", "可链 PubMed；其余用 pmcid 链 PMC。")
print("· 「近 5 年 Nature」：当前样本可直接按年筛，期刊名需规范化后才能实现。")

=== §3.3 元数据 ===
期刊种类数: 65
pmid 覆盖率: 93.00% (93/100)
pub_year 范围: 2022 — 2026
近 5 年（2021+）: 100 篇 (100.00%)


,journal,count
0,Journal of Immigrant and Minority Health,7
1,bioRxiv,7
2,International Journal of Molecular Sciences,5
3,Research Square,5
4,Frontiers in Immunology,3
5,Environmental Science and Pollution Research I...,3
6,Diagnostics,2
7,Animals : an Open Access Journal from MDPI,2
8,Journal of Clinical Medicine,2
9,Frontiers in Oncology,2


,pub_year,count
0,2022,2
1,2023,3
2,2024,3
3,2025,38
4,2026,54



--- 元数据过滤可行性（写入说明文档 §7）---
· 按 journal 过滤：字段完整，但刊名未标准化（如缩写/全称），「Nature」需额外映射表。
· 按 pub_year 近 5 年：pub_year 已可用；精确到月日时用 pub_date。
· pmid 追溯：约 93% 可链 PubMed；其余用 pmcid 链 PMC。
· 「近 5 年 Nature」：当前样本可直接按年筛，期刊名需规范化后才能实现。


## §4 阶段 4：领域内容理解（任务 §2）

> 跑通前请先完成【前置 1/2】【前置 2/2】。分析对象：`df_clean`（97 篇）。逻辑在 `src/domain_analysis.py`。


### 与 §5 关系

§4 与 §5 共用 `all-MiniLM-L6-v2` tokenizer；§4 做分层抽样与领域观察，§5 再做全量 P95/P99 分布图。


### # §4.1 分层抽样（按 abstract token 分桶）

In [32]:

_missing = [k for k in ("df_clean", "TOKENIZER_MODEL_ID", "PATHS") if k not in globals()]
if _missing:
    raise RuntimeError(f"请先运行【前置 1/2】【前置 2/2】，缺少: {_missing}")

import importlib
import domain_analysis as _da

importlib.reload(_da)
from domain_analysis import (
    assign_length_bucket,
    add_abstract_token_len,
    bucket_thresholds,
    export_stratified_markdown,
    load_tokenizer,
    sample_per_bucket,
)

SAMPLES_DIR = PATHS["samples_dir"]

tokenizer = load_tokenizer(TOKENIZER_MODEL_ID)
df4 = add_abstract_token_len(df_clean, tokenizer)
df4 = assign_length_bucket(df4)
thresholds = bucket_thresholds(df4)
samples = sample_per_bucket(df4, n_per_bucket=5)
paths = export_stratified_markdown(samples, SAMPLES_DIR)

print(f"[OK] §4.1 — df_clean {len(df_clean)} 篇 → 每桶 5 篇样例")
print(f"  分桶阈值 (abstract tokens): P33≈{thresholds['p33']:.0f}, P66≈{thresholds['p66']:.0f}")
print(df4["length_bucket"].value_counts().to_string())
print("  已导出:", [str(p) for p in paths])
display(samples[["pmcid", "length_bucket", "abstract_token_len", "journal"]])


[OK] §4.1 — df_clean 97 篇 → 每桶 5 篇样例
  分桶阈值 (abstract tokens): P33≈309, P66≈399
length_bucket
long      33
short     32
medium    32
  已导出: ['/Users/sanxue/Desktop/work/实习/谷歌/02 数据处理/outputs/samples/stratified_short.md', '/Users/sanxue/Desktop/work/实习/谷歌/02 数据处理/outputs/samples/stratified_medium.md', '/Users/sanxue/Desktop/work/实习/谷歌/02 数据处理/outputs/samples/stratified_long.md']


,pmcid,length_bucket,abstract_token_len,journal
0,PMC12882935,short,250,NPJ Digital Medicine
1,PMC12799214,short,248,eLife
2,PMC12882853,short,256,Journal of Immigrant and Minority Health
3,PMC12869580,short,298,Research Square
4,PMC12656031,short,291,"Sensors (Basel, Switzerland)"
5,PMC12882861,medium,335,Environmental Science and Pollution Research I...
6,PMC12825033,medium,357,Frontiers in Public Health
7,PMC12882844,medium,323,Journal of Immigrant and Minority Health
8,PMC12865844,medium,312,Crohn's & Colitis 360
9,PMC12653640,medium,329,Journal of Clinical Medicine


#### 输出分析：
Token indices sequence length ... 586 > 512
原因：样本里至少有 1 篇摘要 用 MiniLM 分词后约 586 tokens，超过该 embedding 模型的 512 上限。
transformers 默认按「要送进模型推理」来检查长度，所以打出警告。

对你当前阶段的影响：没有。§4 的目标是 统计真实长度（包括超过 512 的长摘要），用来：

分 short / medium / long 桶
为 §5、§6 判断「要不要切分、长尾有多少」
586 是有效数字，反而说明：若以后把整段 title+abstract 直接 embedding，必须截断或切块（这正是阶段 5/6 要论证的）。

### §4.2 结构与术语统计

In [33]:

from domain_analysis import imrad_keyword_rates, abbreviation_stats, top_terms

imrad_df = imrad_keyword_rates(df_clean)
abbrev_df = abbreviation_stats(df_clean)
top30 = top_terms(df_clean, top_n=30)

imrad_path = os.path.join(TABLES_DIR, "imrad_keyword_rate.csv")
abbrev_path = os.path.join(TABLES_DIR, "abbrev_density.csv")
terms_path = os.path.join(TABLES_DIR, "abstract_top_terms.csv")
imrad_df.to_csv(imrad_path, index=False)
abbrev_df.to_csv(abbrev_path, index=False)
top30.reset_index().rename(columns={"index": "term"}).to_csv(terms_path, index=True)

print("[OK] §4.2 — IMRaD 关键词出现率、缩写密度、高频词 Top-30")
display(imrad_df)
display(abbrev_df.tail(1))
display(top30.head(15))
print(f"已保存: {imrad_path}\n       {abbrev_path}\n       {terms_path}")


[OK] §4.2 — IMRaD 关键词出现率、缩写密度、高频词 Top-30


,section,match_count,match_rate
0,background,8,0.0825
1,methods,20,0.2062
2,results,53,0.5464
3,conclusion,11,0.1134
4,all_four,4,0.0412


,pmcid,abbrev_count,word_count,abbrev_per_100_words
97,__summary__,8,245,3.76


patients      95
study         73
using         72
analysis      68
risk          57
between       53
health        53
higher        49
associated    47
data          47
cell          47
treatment     42
cells         41
high          40
may           40
Name: count, dtype: int64

已保存: /Users/sanxue/Desktop/work/实习/谷歌/02 数据处理/outputs/tables/imrad_keyword_rate.csv
       /Users/sanxue/Desktop/work/实习/谷歌/02 数据处理/outputs/tables/abbrev_density.csv
       /Users/sanxue/Desktop/work/实习/谷歌/02 数据处理/outputs/tables/abstract_top_terms.csv


### §4.3 人工观察笔记


| 观察项 | 结论 |
|---|---|
| 语言风格 / 信息密度 | 正式学术英语；信息密度高，单篇摘要常含研究设计、样本量、主要发现 |
| IMRaD 结构是否常见 | **Results** 关键词最常见（~55%）；四段标题全齐仅 ~4%。多数摘要为连贯段落而非显式小标题 |
| 缩写是否普遍 | 平均每百词约 **3.8** 个大写缩写（EGFR、PCI 等）；检索/query 需容忍缩写与全称混用 |
| 高频词（Top-5） | patients, study, using, analysis, risk（见 `abstract_top_terms.csv`） |
| 同义表述举例 1 | ICI / immune checkpoint inhibitors；HCC / hepatocellular carcinoma（PMC12882918） |
| 同义表述举例 2 | PDCI（全称 Parkinson’s disease with cognitive impairment）与文中缩写并存（PMC12872877） |
| 对 RAG prompt / 评估的启示 | 评估集应覆盖短/长摘要；问法宜具体；勿假设摘要必有 METHODS/RESULTS 小标题 |


## §5 阶段 5：文本特征量化（任务 §3）

> 前置：【前置 1/2】【前置 2/2】；与 §4 共用同一 tokenizer（`TOKENIZER_MODEL_ID`）。

### 本节目标

- 对 **title / abstract / title+abstract / body** 统计 **token** 长度
- 输出 **P50/P75/P95/P99/max** → `outputs/tables/token_percentiles.csv`
- 绘制 **ECDF**（摘要 + 检索单元）→ `outputs/figures/token_dist_abstract.png`
- 对照 embedding **512** 上限，为阶段 6 分割策略提供数字依据


In [3]:
# §5.1–5.3 token 分布与可视化
_missing = [k for k in ("df_clean", "TOKENIZER_MODEL_ID", "PATHS", "TABLES_DIR") if k not in globals()]
if _missing:
    raise RuntimeError(f"请先运行【前置 1/2】【前置 2/2】，缺少: {_missing}")

import importlib
import token_stats as _ts

importlib.reload(_ts)
from domain_analysis import load_tokenizer
from token_stats import run_stage5_token_report, embedding_fit_summary

FIGURES_DIR = PATHS["figures_dir"]
tokenizer = load_tokenizer(TOKENIZER_MODEL_ID)
r5 = run_stage5_token_report(
    df_clean,
    tokenizer,
    tables_dir=TABLES_DIR,
    figures_dir=FIGURES_DIR,
    model_id=TOKENIZER_MODEL_ID,
)
df5 = r5["df5"]
print("[OK] §5 — 分位数表与 ECDF 已写入 outputs/")
display(r5["percentiles"])
fit = r5["embedding_fit"]
print("\n与 512 token 上限对照（检索单元 = title+abstract）:")
print(f"  P95(retrieval): {fit.get('retrieval_p95')}")
print(f"  P99(retrieval): {fit.get('retrieval_p99')}")
print(f"  超过 512 占比: {fit.get('retrieval_over_limit_rate', 0):.2%}")
print(f"  图: {r5['figure_path']}")
print(f"  表: {r5['percentiles_path']}")


[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (12770 > 10000). Running this sequence through the model will result in indexing errors


[OK] §5 — 分位数表与 ECDF 已写入 outputs/


,field,n,mean,min,p50,p75,p95,p99,max
0,title,97,23.02,6.0,22.0,27.0,36.2,49.08,51.0
1,abstract,97,377.82,99.0,354.0,425.0,587.4,977.36,986.0
2,title+abstract,97,400.85,110.0,374.0,455.0,617.2,1009.00,1009.0
3,body,97,9738.39,1499.0,7889.0,11928.0,20574.0,26213.36,54542.0



与 512 token 上限对照（检索单元 = title+abstract）:
  P95(retrieval): 617.1999999999998
  P99(retrieval): 1009.0
  超过 512 占比: 14.43%
  图: /Users/sanxue/Desktop/work/实习/谷歌/02 数据处理/outputs/figures/token_dist_abstract.png
  表: /Users/sanxue/Desktop/work/实习/谷歌/02 数据处理/outputs/tables/token_percentiles.csv


## §6 阶段 6：制定文本分割策略（任务 §4）

> 前置：【前置 1/2】【前置 2/2】；建议已跑 **§5**（`df5` 含 `retrieval_token_len`）。  
> 实现：`src/chunk_strategy.py`（定稿参数 + LangChain demo）。**不做全库入库**，仅验证策略。

### 定稿要点

- **检索单元**：`title + abstract`
- **≤512 tokens**：单 chunk（约 85%）
- **>512 tokens**：`RecursiveCharacterTextSplitter`（400 / overlap 80）
- **body**：单独 sliding window（512 / overlap 80）；二期全文检索再用
- **无 abstract**：丢弃（阶段 3 已执行）；**低质量 flag** 仅标记，验证期无 short_abstract


## §6.1 定稿配置 + 97 篇 retrieval 切块汇总

In [6]:

_missing = [k for k in ("df_clean", "TOKENIZER_MODEL_ID", "PATHS", "TABLES_DIR") if k not in globals()]
if _missing:
    raise RuntimeError(f"请先运行【前置 1/2】【前置 2/2】，缺少: {_missing}")

import importlib
import chunk_strategy as _cs

importlib.reload(_cs)
from domain_analysis import load_tokenizer
from chunk_strategy import (
    ChunkStrategyConfig,
    chunk_retrieval_row,
    chunk_body_text,
    print_strategy_banner,
    summarize_retrieval_chunks,
    config_as_dict,
)

tokenizer = load_tokenizer(TOKENIZER_MODEL_ID)
CFG = ChunkStrategyConfig()
print_strategy_banner(CFG)

chunk_summary = summarize_retrieval_chunks(df_clean, tokenizer, config=CFG)
summary_path = os.path.join(TABLES_DIR, "chunk_strategy_summary.csv")
chunk_summary.to_csv(summary_path, index=False)

n_single = int((chunk_summary.iloc[:-1]["n_chunks"] == 1).sum())
n_multi = int((chunk_summary.iloc[:-1]["n_chunks"] > 1).sum())
print(f"\n[OK] §6.1 — retrieval 切块: 单块 {n_single} 篇, 多块 {n_multi} 篇 (共 {len(df_clean)} 篇)")
display(chunk_summary.iloc[:-1].query("n_chunks > 1").head(10))
print(f"已保存: {summary_path}")

# 保存 JSON 配置供未来 ingest 脚本读取
import json
cfg_path = os.path.join(TABLES_DIR, "chunk_strategy_config.json")
with open(cfg_path, "w", encoding="utf-8") as f:
    json.dump(config_as_dict(CFG), f, ensure_ascii=False, indent=2)
print(f"配置快照: {cfg_path}")


=== 阶段 6：定稿分割策略（供未来 RAG ingest） ===
  retrieval_unit: title+abstract
  retrieval_token_limit: 512
  retrieval_chunk_size: 400
  retrieval_chunk_overlap: 80
  body_chunk_size: 512
  body_chunk_overlap: 80
  drop_missing_abstract: True
  drop_short_abstract: False
  short_abstract_char_threshold: 50
notes:
  - 主体 ~85% title+abstract ≤512：单 chunk
  - 长尾 ~14% >512：RecursiveCharacterTextSplitter
  - body 一律 sliding_window，与摘要分开
  - 无 abstract：丢弃并记录 pmcid

[OK] §6.1 — retrieval 切块: 单块 83 篇, 多块 14 篇 (共 97 篇)


,pmcid,retrieval_token_len,n_chunks,strategy
9,PMC11939599,530,2,sliding_window
15,PMC12295483,695,3,sliding_window
23,PMC12650391,946,3,sliding_window
41,PMC12815873,534,2,sliding_window
44,PMC12832483,547,2,sliding_window
49,PMC12869397,634,4,sliding_window
54,PMC12869590,1088,4,sliding_window
58,PMC12871179,639,3,sliding_window
59,PMC12871241,626,2,sliding_window
64,PMC12872877,666,3,sliding_window


已保存: /Users/sanxue/Desktop/work/实习/谷歌/02 数据处理/outputs/tables/chunk_strategy_summary.csv
配置快照: /Users/sanxue/Desktop/work/实习/谷歌/02 数据处理/outputs/tables/chunk_strategy_config.json


## §6.2 demo：1 条长尾 retrieval + 1 条 body（各展示首尾 chunk）

In [7]:

if "chunk_summary" not in globals():
    raise RuntimeError("请先运行 §6.1")

# 长尾 retrieval：选 n_chunks 最大的一篇
sub = chunk_summary.iloc[:-1]
demo_pid = sub.loc[sub["n_chunks"].idxmax(), "pmcid"]
row = df_clean[df_clean["pmcid"] == demo_pid].iloc[0]
rchunks = chunk_retrieval_row(row, tokenizer, config=CFG)
print(f"--- retrieval demo: {demo_pid} → {len(rchunks)} chunks ---")
for c in rchunks[:2]:
    print(f"  [{c.chunk_index+1}/{c.chunk_count}] tokens={c.token_len} strategy={c.strategy}")
    print(f"  {c.text[:200].replace(chr(10), ' ')}…")

# body demo：第一篇
row0 = df_clean.iloc[0]
bchunks = chunk_body_text(row0["body"], tokenizer, pmcid=row0["pmcid"], config=CFG)
print(f"\n--- body demo: {row0['pmcid']} → {len(bchunks)} chunks (仅演示，首轮 RAG 可不索引 body) ---")
print(f"  chunk[0] tokens={bchunks[0].token_len}, chunk[-1] tokens={bchunks[-1].token_len}")


--- retrieval demo: PMC12869397 → 4 chunks ---
  [1/4] tokens=18 strategy=sliding_window
  Changes in Visual Attention Patterns for Detection Tasks due to Dependencies on Signal and Background Spatial Frequencies…
  [2/4] tokens=400 strategy=sliding_window
  We aim to investigate the impact of image and signal properties on visual attention mechanisms during a signal detection task in digital images. The application of insight yielded from this work spans…

--- body demo: PMC8774754 → 8 chunks (仅演示，首轮 RAG 可不索引 body) ---
  chunk[0] tokens=395, chunk[-1] tokens=387


# 小规模数据集验证阶段最终策略

> 数据：**100 篇**抽样 → 去掉 **3 篇无摘要** → **97 篇**用于分析（`sample_clean.jsonl`）。  
> 下文数字均来自本 notebook **§4～§6** 已跑通结果；全量入库时直接复用 `outputs/tables/chunk_strategy_config.json`。

---

## 1. 检索时搜什么？

**用「标题 + 摘要」**（`title+abstract`），不用单独摘要、也不和正文混在同一条规则里。

- 理由：和正文比，长度可控；相对「只用摘要」，多标题只多很少 token（>512 占比约多 1%）。

---

## 2. 摘要怎么切块？（嵌入上限 512 token）

模型口径：`sentence-transformers/all-MiniLM-L6-v2`（与后续向量模型一致）。

| 情况 | 篇数（97 篇里） | 做法 |
|------|----------------|------|
| 标题+摘要 **≤ 512 token** | **83 篇** | **整块** → 1 条向量 |
| 标题+摘要 **> 512 token** | **14 篇** | **滑动窗口**：每块约 **400** token，相邻重叠 **80** |
| 合计 chunk 数 | — | **123** 个 retrieval chunk（`chunk_strategy_summary.csv`） |

**不是**每篇都滑窗——大约 **85% 单块、15% 多块**；入库时按规则自动选，**查询时不再临时数 token**。

---

## 3. 正文 body 怎么办？

- 正文普遍 **上万 token**，必须按 **512 / 重叠 80** 切块（与摘要规则分开）。
- **首轮 RAG 建议：只建索引 title+abstract**，正文二期再索引（本 notebook §6.2 仅演示 body 切块）。

---

## 4. 清洗与质量

| 规则 | 验证期结果 |
|------|------------|
| **无 abstract** | **丢弃**（3 篇），全量用 `--skip-no-abstract` |
| **摘要过短** 等低质量 | 只 **打 flag**，验证期 **0 篇** 需排除 |
| 按论文章节（IMRaD 标题）切 | **备选**，本批标题极少，**默认不用** |

---

## 5. 全量期（明日，外接盘）

- 数据根：`/Volumes/Lexar/med-llm-rag-datasets`（`source scripts/setup_full_data_env.sh`）
- 产出：slim jsonl（**不含 body 列**）+ pmcid 索引；正文仍从 XML 按需回查
- 切块参数：**勿改** `chunk_strategy_config.json`，除非全量统计与验证期差异很大

---

## 6. 相关产出（可随仓库提交）

| 文件 | 含义 |
|------|------|
| `outputs/tables/token_percentiles.csv` | §5 分位数 |
| `outputs/figures/token_dist_abstract.png` | §5 ECDF 图 |
| `outputs/tables/chunk_strategy_summary.csv` | §6 每篇 chunk 数 |
| `outputs/tables/chunk_strategy_config.json` | ingest 参数快照 |
| `docs/RAG数据分析与设计说明.md` | 正式说明文档 §1～§6 |

**阶段 02 范围**：数据分析 + 切块策略定稿；**不做** Chroma 入库与 LLM 问答。
